# 第 2 章：統計學習 — 2.1 什麼是統計學習？

> **課本**：James, Witten, Hastie, Tibshirani (2023), *An Introduction to Statistical Learning with Applications in Python*, Springer.
> **對應章節**：2.1 What Is Statistical Learning?（第 25–37 頁）
> **難度**：★★☆☆☆ 入門概念
> **關鍵字**：統計學習模型、預測 vs 推論、可約/不可約誤差、參數化 vs 非參數化、監督 vs 非監督學習、迴歸 vs 分類

## 一、動機：Advertising 資料集

想像我們是一間被聘請的統計顧問公司——客戶想知道廣告支出與產品銷售量之間的關聯。
`Advertising` 資料集包含了 200 個市場中，三種廣告媒體的預算以及對應的銷售量。

In [ ]:
# ============================================================
# Cell 1：Advertising 資料集探索
# 對應課本 Figure 2.1 — 三種廣告媒體與銷售量的關係
# 課本核心案例：我們是統計顧問，客戶想知道廣告支出與銷售量的關聯
# ============================================================
import pandas as pd                          # 表格資料處理（DataFrame）
import numpy as np                           # 數值運算與陣列操作
import matplotlib.pyplot as plt              # 繪圖：散佈圖、趨勢線

# 讀取 Advertising 資料集（200 個市場的廣告支出與銷售量）
# index_col=0：第一欄是市場編號，作為列索引而非資料欄
ad = pd.read_csv('data/Advertising.csv', index_col=0)

# .shape → (200, 4)：200 個市場 × 4 個變數
print(f"維度: {ad.shape}")

# .head() → 查看前 5 筆，快速確認欄位名稱與數值範圍
print(ad.head())

# .describe() → 每個數值欄的統計摘要：
#   count（樣本數）、mean（平均）、std（標準差）、
#   min/25%/50%/75%/max（最小值、四分位數、最大值）
# 這是資料探索的標準第一步——了解各變數的尺度與分散程度
print(ad.describe())

# 📋 變數說明：
#   TV        — 電視廣告預算（千美元）  ← 預測子 X₁
#   radio     — 廣播廣告預算（千美元）  ← 預測子 X₂
#   newspaper — 報紙廣告預算（千美元）  ← 預測子 X₃
#   sales     — 銷售量（千單位）       ← 反應變數 Y（我們想預測的目標）

| 變數 | 意義 |
|------|------|
| `TV` | 電視廣告預算（千美元） |
| `radio` | 廣播廣告預算（千美元） |
| `newspaper` | 報紙廣告預算（千美元） |
| `sales` | 銷售量（千單位）— **反應變數 Y** |

In [ ]:
# ============================================================
# Cell 2：繪製 Figure 2.1 — 三種媒體 vs 銷售量的散佈圖與迴歸線
# 目的：視覺化判斷「哪個廣告媒體與銷售量最有關聯」
# 方法：對每個媒體分別擬合簡單線性迴歸（sales = β₀ + β₁ × Media）
# ============================================================
from sklearn.linear_model import LinearRegression  # 簡單線性迴歸（最小平方法）

# plt.subplots(1, 3)：建立 1 列 × 3 欄的子圖佈局
# figsize=(15, 5)：整張圖寬 15 吋、高 5 吋（每個子圖約 5×5）
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 逐一處理三種媒體：TV、radio、newspaper
for ax, var in zip(axes, ['TV', 'radio', 'newspaper']):
    # --- 散佈圖：x = 該媒體的廣告支出，y = 銷售量 ---
    # alpha=0.5：點半透明 → 可以看出重疊區域的密度
    # edgecolor='k'：黑色邊框，每個點輪廓清楚
    # facecolor='red'：紅色填充（ISLR 課本經典配色）
    ax.scatter(ad[var], ad['sales'], alpha=0.5, edgecolor='k', facecolor='red')
    
    # --- 擬合簡單線性迴歸 ---
    # 公式：sales = β₀ + β₁ × var
    # X 需為二維 (n_samples, n_features)，所以用雙層 [[]]
    # y 只要一維 (n_samples,)
    X = ad[[var]].values      # 形狀 (200, 1) — scikit-learn 要求二維輸入
    y = ad['sales'].values     # 形狀 (200,)  — 一維目標變數
    
    model = LinearRegression().fit(X, y)  # 最小平方法估計 β₀ 和 β₁
    
    # --- 畫出迴歸線（藍色實線）---
    # np.linspace：在 X 的範圍內均勻取 100 個點，畫出平滑直線
    x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
    
    # model.predict()：用估計出的 β₀ + β₁·X 計算每個 X 點的預測 ŷ
    ax.plot(x_line, model.predict(x_line), 'b-', linewidth=2)
    
    # 加上 x 軸和 y 軸標籤
    ax.set_xlabel(var)
    ax.set_ylabel('Sales')
    
    # 輸出迴歸係數：截距 β₀ 和斜率 β₁
    # model.intercept_ = β₀（廣告支出=0 時的預期銷售量）
    # model.coef_[0]   = β₁（廣告支出每增加 1 單位，銷售量平均變動量）
    print(f"Sales ~ {var}: 截距={model.intercept_:.2f}, 斜率={model.coef_[0]:.4f}")

# 整張圖的總標題
fig.suptitle('Figure 2.1: Advertising Data', fontsize=14)
plt.tight_layout()  # 自動調整子圖間距，避免標籤重疊
plt.show()

# ════════════════ 觀察重點 ════════════════
# ✅ TV：散佈圖右上趨勢明顯，迴歸線斜率為正且陡峭
#    → 電視廣告與銷售量有強烈正相關
# ✅ radio：也有上升趨勢，但點較 TV 分散
#    → 廣播廣告也有正向影響，但關係稍弱
# ⚠️ newspaper：點非常分散，迴歸線幾乎水平
#    → 報紙廣告與銷售量幾乎無關
# ═══════════════════════════════════════════

> **觀察**：TV 和 radio 與 sales 有明顯的正相關，而 newspaper 的相關性較弱。

---

## 二、統計學習模型：Y = f(X) + ε

課本用一個極其簡潔的方程式概括了整個統計學習的框架：

$$Y = f(X) + \epsilon$$

| 符號 | 含義 |
|------|------|
| $X = (X_1, X_2, \dots, X_p)$ | **輸入變數**（預測子、特徵、自變數） |
| $Y$ | **輸出變數**（反應變數、應變數） |
| $f$ | **未知的真實函數**——X 提供關於 Y 的「系統性資訊」 |
| $\epsilon$ | **隨機誤差項**——與 X 獨立、均值為零 |

### 用圖形理解 Y = f(X) + ε

In [ ]:
# ============================================================
# Cell 3：Figure 2.2 — 視覺化解釋 Y = f(X) + ε
# 目的：用圖說「觀測值 = 系統資訊 + 隨機雜訊」的分解
# 資料：Income1.csv — 教育年數 vs 年收入
# ============================================================
from sklearn.preprocessing import PolynomialFeatures  # 多項式特徵生成

income1 = pd.read_csv('data/Income1.csv', index_col=0)

X = income1[['Education']].values  # 預測子：教育年數（年）
y = income1['Income'].values       # 反應變數：年收入（千美元）

# --- 用三次多項式擬合 f(X) ---
# poly.fit_transform：將 X 轉換為 [1, X, X², X³]
# 為什麼用 degree=3？因為收入增長通常非線性：教育初期收益大、後期趨緩
# f(X) = β₀ + β₁·Education + β₂·Education² + β₃·Education³
poly = PolynomialFeatures(degree=3)
model = LinearRegression().fit(poly.fit_transform(X), y)

# 建立左右對比圖形
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ===== 左圖：只有觀測資料（我們看到的）=====
ax1.scatter(income1['Education'], income1['Income'], c='red', s=50)
ax1.set_xlabel('Education')
ax1.set_ylabel('Income')
ax1.set_title('Left: Observed Data')

# ===== 右圖：疊加 f̂(X) 曲線 + 誤差線 =====
# 先畫資料點（zorder=5 確保點在最上層）
ax2.scatter(income1['Education'], income1['Income'], c='red', s=50, zorder=5)

# 產生 200 個均勻點，畫出平滑的 f̂(X) 曲線（藍色）
x_curve = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
ax2.plot(x_curve, model.predict(poly.transform(x_curve)), 'b-', lw=2, 
         label='f(Education)')  # 這是統計學習估計出的 f̂

# 畫出每個點的誤差 ε = Y - f̂(X)（黑色垂直線）
# 垂直線長度 = 該觀測點的隨機誤差大小
y_pred = model.predict(poly.transform(X))
for i in range(len(X)):
    ax2.plot([X[i,0], X[i,0]], [y_pred[i], y[i]], 'k-', lw=0.8, alpha=0.6)

ax2.set_xlabel('Education')
ax2.set_ylabel('Income')
ax2.set_title('Right: f(X) + Error Terms (ε)')
ax2.legend()

# 驗證：ε 的理論均值應為 0（無系統性偏差）
errors = y - y_pred
print(f"誤差均值 ≈ {errors.mean():.6f}（應趨近 0，代表模型無系統偏差）")

plt.show()

# ════════════════ 關鍵洞察 ════════════════
# 🔵 藍色曲線 = f̂(X)：系統性資訊（教育如何影響收入）
# ⬛ 黑色垂直線 = ε：無法用教育解釋的收入差異
#    （天賦、產業、運氣...這些不在模型中的因素）
# 📐 每個紅點 = 藍色曲線值 + 黑色誤差線
# ═══════════════════════════════════════════

> 藍色曲線是真正 f 的估計值（真正的 f 未知！），黑色垂直線是 ε。有些點在曲線上方、有些在下方，但整體誤差均值約為零。

---

## 三、為什麼要估計 f？預測 vs 推論

### 3.1 預測（Prediction）

$$\hat{Y} = \hat{f}(X)$$

當我們只在意預測的**準確度**，不在意 $\hat{f}$ 的具體形式時，$\hat{f}$ 可視為一個**黑盒子**（black box）。

**範例**：預測病人對某藥物的不良反應風險 → 只要預測準，不需要解釋「為什麼」。

預測誤差可分解為：

$$E(Y - \hat{Y})^2 = \underbrace{[f(X) - \hat{f}(X)]^2}_{\text{可約誤差 (Reducible)}} + \underbrace{\text{Var}(\epsilon)}_{\text{不可約誤差 (Irreducible)}}$$

In [ ]:
# ============================================================
# Cell 4：可約誤差 vs 不可約誤差 — 預測準確度的理論極限
# 公式：E[(Y - Ŷ)²] = [f(X) - f̂(X)]² + Var(ε)
#       總誤差       =   可約誤差        + 不可約誤差
# 核心問題：為什麼任何模型都有無法消除的預測誤差？
# ============================================================
np.random.seed(42)  # 固定隨機種子 → 每次執行結果相同（教學可重現性）

# --- 定義「真實但未知」的函數 f(X) ---
# 真實 f 是二次函數：f(X) = 2 + 3X + 0.5X²
# 注意：在真實世界中我們永遠不知道這個函數！
def true_f(X):
    return 2 + 3*X + 0.5*X**2

# --- 產生模擬資料：Y = f(X) + ε ---
n = 100                                          # 100 個觀測點
X_sim = np.random.uniform(0, 10, n)              # X 在 [0, 10] 均勻分布
epsilon = np.random.normal(0, 3, n)               # ε ~ N(0, 3²)，不可約誤差的來源
Y_sim = true_f(X_sim) + epsilon                   # Y = f(X) + ε

# ===== Model A：線性模型（設定錯誤 → 可約誤差大）=====
# 用直線擬合曲線 → 必然有偏差，這個偏差就是「可約」的部分
model_A = LinearRegression().fit(X_sim.reshape(-1, 1), Y_sim)
Y_pred_A = model_A.predict(X_sim.reshape(-1, 1))

# ===== Model B：二次模型（正確設定 → 可約誤差小）=====
# 用二次函數的基底去擬合二次函數的真實 f
# PolynomialFeatures(degree=2)：產生 [1, X, X²] 三個特徵
poly2 = PolynomialFeatures(degree=2)
model_B = LinearRegression().fit(poly2.fit_transform(X_sim.reshape(-1, 1)), Y_sim)
Y_pred_B = model_B.predict(poly2.transform(X_sim.reshape(-1, 1)))

# --- 計算可約誤差：f̂ 與真實 f 的差距平方平均 ---
# 可約誤差 = mean((true_f(X) - f̂(X))²)
# 這是「模型選錯形式」所付出的代價，理論上可以透過改良模型來消除
reducible_A = np.mean((true_f(X_sim) - Y_pred_A)**2)
reducible_B = np.mean((true_f(X_sim) - Y_pred_B)**2)

# --- 不可約誤差 = Var(ε) = 9（我們設定的 σ²=3²）---
# 這是資料本身固有的雜訊，任何模型都無法降低！
irreducible = np.var(epsilon)

print(f"Var(ε) — 不可約誤差:      {irreducible:.3f}")
print(f"Model A (線性) 可約誤差:  {reducible_A:.3f}  總 MSE ≈ {reducible_A + irreducible:.3f}")
print(f"Model B (二次) 可約誤差:  {reducible_B:.3f}  總 MSE ≈ {reducible_B + irreducible:.3f}")
print()
print("→ 不可約誤差是預測準確度的「理論上界」")
print("→ Model B 的可約誤差接近零，因正確設定了 f 的函數形式")
print("→ Model A 的額外誤差來自「模型設定錯誤」（用直線擬合曲線）")

# ════════════════ 核心教訓 ════════════════
# 🎯 即使完美估計 f（Model B），預測誤差也無法低於 Var(ε)
# 📉 Model A 的可約誤差很大 → 可透過改善模型來降低
# 🔑 實務中我們不知道真實 f → 選對模型形式至關重要
# ═══════════════════════════════════════════

### 3.2 推論（Inference）

當目標是**理解 X 與 Y 之間的關係**時：

- 🔍 哪些預測子與反應變數有關聯？
- 📈 關聯的方向與強度為何？（正向？負向？）
- 📐 關係可否用簡單線性方程式概括，還是需要更複雜的形式？

**範例**：哪些廣告媒體（TV/radio/newspaper）真正帶動銷售？增加 TV 預算能多賣多少？

### 3.3 預測 vs 推論的選擇指南

| 目標 | 偏好模型 | 原因 |
|------|---------|------|
| 純預測 | 靈活模型（黑盒子 OK） | 追求最高準確度 |
| 純推論 | 簡單、可解釋模型 | 需要理解變數關係 |
| 兩者兼顧 | 適中模型 | 取捨靈活度與可解釋性 |

---

## 四、如何估計 f？參數化 vs 非參數化方法

### 4.1 參數化方法（Parametric）

**兩步驟流程**：

1. **假設 f 的函數形式**：例如線性模型
   $$f(X) = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p$$

2. **用訓練資料估計參數**：例如最小平方法（least squares）

In [ ]:
# ============================================================
# Cell 5：參數化方法 — 線性迴歸（Income2 資料集）
# 資料：Income2.csv — 教育年數 + 年資 → 年收入（兩個預測子）
# 模型：Income = β₀ + β₁·Education + β₂·Seniority + ε
# 參數化核心：先假設 f 是線性的，再估計 β₀、β₁、β₂
# ============================================================
income2 = pd.read_csv('data/Income2.csv', index_col=0)

# 兩個特徵：教育程度和年資（各觀測點的 X 值）
X2 = income2[['Education', 'Seniority']].values  # 形狀 (n, 2)
y2 = income2['Income'].values                     # 形狀 (n,) — 目標：年收入

# --- 參數化方法：線性迴歸 ---
# 假設 f(X₁, X₂) = β₀ + β₁·X₁ + β₂·X₂
# 這個假設大幅簡化問題：只需估計 3 個參數（β₀, β₁, β₂）
model_linear = LinearRegression().fit(X2, y2)

print("=== 參數化：線性迴歸 ===")
# model.intercept_ = β₀（基準收入，當教育=0 且年資=0 時）
# model.coef_[0]   = β₁（教育年數每增加 1 年，收入變動量）
# model.coef_[1]   = β₂（年資每增加 1 年，收入變動量）
print(f"Income ≈ {model_linear.intercept_:.2f} + "
      f"{model_linear.coef_[0]:.2f}·Education + "
      f"{model_linear.coef_[1]:.2f}·Seniority")
print(f"僅需估計 3 個參數（β₀, β₁, β₂）— 模型極簡")
print()
print("係數解讀：")
print(f"  ├─ 教育年數每增加 1 年 → 收入平均增加 {model_linear.coef_[0]:.0f} 單位")
print(f"  └─ 年資每增加 1 年     → 收入平均增加 {model_linear.coef_[1]:.0f} 單位")
print()
print("✅ 優點：模型簡單，參數直接可解釋（每個 β 都有明確意義）")
print("⚠️ 缺點：真實關係可能非線性 → 預測可能不準")
print("   → 如果真實 f 不是線性的，參數化方法會產生系統偏差

**優點**：簡化問題（只需估計少數參數）
**缺點**：模型假設可能與真實 f 差距甚遠 → 預測不佳

### 4.2 非參數化方法（Non-parametric）

**不假設 f 的特定函數形式**，而是讓資料「自己說話」。例如：
- K-最近鄰（KNN）
- 平滑薄板樣條（thin-plate spline）
- 決策樹

In [ ]:
# ============================================================
# Cell 6：非參數化方法 — KNN 迴歸 vs 線性迴歸
# KNN 核心概念：不對 f 做任何函數形式假設
# Ŷ(x) = 距離 x 最近的 k 個訓練點之 Y 值的平均
# 這是一個「讓資料自己說話」的方法
# ============================================================
from sklearn.neighbors import KNeighborsRegressor  # KNN 迴歸
from sklearn.metrics import mean_squared_error     # 均方誤差 MSE

# k=3：用最近的 3 個鄰居的平均值來預測
# k 越小 → 模型越靈活（但可能過度擬合雜訊）
# k 越大 → 模型越平滑（但可能忽略真實模式）
model_knn = KNeighborsRegressor(n_neighbors=3).fit(X2, y2)

# --- 比較兩個模型在「訓練資料」上的表現 ---
y_pred_linear = model_linear.predict(X2)
y_pred_knn = model_knn.predict(X2)

print(f"訓練 MSE — Linear (3 參數):    {mean_squared_error(y2, y_pred_linear):.3f}")
print(f"訓練 MSE — KNN(k=3, 無參數):   {mean_squared_error(y2, y_pred_knn):.3f}")
print()
print("對比重點：")
print("  ├─ 線性迴歸：強烈假設 f 是線性 → 3 個參數 → 高可解釋性")
print("  ├─ KNN(k=3)：不假設函數形式 → 無簡化 → 讓資料說話")
print("  └─ ⚠️ 注意：這是「訓練誤差」！不代表在「新資料」上的表現")
print("         → KNN 可能只是記住了訓練資料（過度擬合）")

**優點**：可擬合極廣泛的函數形狀
**缺點**：需要大量觀測值才能準確估計（因為沒有將問題化簡為參數）

### 4.3 過度擬合（Overfitting）的危險

In [ ]:
# ============================================================
# Cell 7：過度擬合的危險 — 三種平滑度的 Spline 對比
# 目的：直觀展示「太簡單 → 欠擬合」→「適中」→「太複雜 → 過度擬合」
# Spline（樣條）：一種靈活的非參數化方法，用分段多項式擬合資料
# 對應課本 Figure 2.5 / 2.6
# ============================================================
from sklearn.preprocessing import SplineTransformer  # B-spline 基底函數

np.random.seed(42)

# --- 產生模擬資料 ---
X_s = np.sort(np.random.uniform(0, 1, 30))  # 30 個觀測點，均勻分布在 [0,1]

# 真實 f 是一個有週期性波動的函數（我們假裝不知道！）
true_f_s = lambda x: np.sin(2*np.pi*x) + 0.3*np.cos(6*np.pi*x)

# Y = f(X) + ε，雜訊 σ=0.2（相對較小，讓過度擬合更明顯）
Y_s = true_f_s(X_s) + np.random.normal(0, 0.2, 30)

# 用於畫平滑曲線的密集 X（500 個點，不參與訓練）
X_plot = np.linspace(0, 1, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 三個不同節點數量的樣條
# n_knots（節點數）：控制 spline 的靈活度
#   節點少 = 曲線簡單（可能欠擬合）
#   節點多 = 曲線複雜（可能過度擬合）
for ax, n_k, title in zip(axes, [3, 8, 25],
    ['Smooth (3 節點) → 欠擬合',
     'Moderate (8 節點) → 良好',
     'Rough (25 節點) → 過度擬合！']):
    
    # degree=3：三次樣條（最常用，確保曲線及一二階導數連續）
    spline = SplineTransformer(n_knots=n_k, degree=3)
    
    # 將 X 轉換為 spline 基底，再用線性迴歸擬合
    # 等同於用分段三次多項式來逼近真實 f
    m = LinearRegression().fit(spline.fit_transform(X_s.reshape(-1,1)), Y_s)
    
    # --- 繪圖 ---
    ax.scatter(X_s, Y_s, c='red', s=30, zorder=5)          # 訓練資料點
    ax.plot(X_plot, m.predict(spline.transform(X_plot)),    # 擬合曲線（藍色）
            'b-', lw=2)
    # 黑色虛線：真實 f（參考用，實務中看不到）
    ax.plot(X_plot, true_f_s(X_plot), 'k--', lw=1, alpha=0.4, label='True f')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Figure 2.5/2.6 概念：非參數化方法的平滑度取捨', fontsize=14)
plt.tight_layout()
plt.show()

# ════════════════ 觀察關鍵 ════════════════
# 🔴 左圖（3 節點）：曲線太簡單 → 無法捕捉雙峰波動 → 欠擬合
#    → 偏差高、變異低
# 🟢 中圖（8 節點）：捕捉主要趨勢，沒被雜訊誤導 → 良好平衡
#    → 偏差與變異的甜蜜點
# 🔵 右圖（25 節點）：連雜訊都完美擬合 → 劇烈波動無意義
#    → 在新資料上預測效果極差 = 過度擬合！
#    → 訓練誤差接近零 ≠ 模型好
# ═══════════════════════════════════════════

> 右圖：訓練誤差為零（完美擬合訓練資料），但在新資料上的預測效果極差——這就是**過度擬合**！

---

## 五、靈活度與可解釋性的取捨

課本 Figure 2.7 展示了重要權衡：

```
靈活度低 ←——————————————————→ 靈活度高
可解釋性高                             可解釋性低

Least Squares ── Lasso ── GAM ── Trees ── Bagging/Boosting ── SVM ── Deep Learning
```

In [ ]:
# ============================================================
# Cell 8：靈活度 vs 可解釋性 — 不同多項式階數的視覺化
# 目的：用 d=1（線性）→ d=3（適中）→ d=15（過度擬合）展示取捨
# 對應課本 Figure 2.7
# 核心問題：「更準的預測」和「更易懂的模型」，你選哪一個？
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
np.random.seed(123)

# --- 產生模擬資料 ---
# 真實 f 是 sin(2πx)（正弦波，非線性但平滑）
X_f = np.sort(np.random.uniform(0, 1, 50))           # 50 個觀測點
Y_f = np.sin(2*np.pi*X_f) + np.random.normal(0, 0.3, 50)  # Y = f(X) + ε

# 三種不同靈活度的多項式回歸
for ax, deg, title in zip(axes, [1, 3, 15],
    ['d=1：線性（低靈活度／高可解釋）',
     'd=3：三次多項式（適中）',
     'd=15：15 次多項式（高靈活度／低可解釋）→ 過度擬合']):
    
    # PolynomialFeatures(degree=d)：產生 [X, X², ..., X^d]
    # d 越大 → 模型參數越多 → 靈活性越高 → 越容易過度擬合
    poly = PolynomialFeatures(degree=deg)
    m = LinearRegression().fit(poly.fit_transform(X_f.reshape(-1,1)), Y_f)
    
    ax.scatter(X_f, Y_f, c='red', s=20)                     # 訓練資料點
    Xp = np.linspace(0, 1, 500).reshape(-1, 1)              # 平滑曲線用
    ax.plot(Xp, m.predict(poly.transform(Xp)), 'b-', lw=2)  # 擬合曲線
    # 真實函數（黑虛線）
    ax.plot(Xp, np.sin(2*np.pi*Xp), 'k--', lw=1, alpha=0.4, label='True f')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('靈活度 vs 可解釋性（Figure 2.7 概念）', fontsize=14)
plt.tight_layout()
plt.show()

# ════════════════ 模型光譜 ════════════════
# 靈活度低 ──────────────────────────→ 靈活度高
#   可解釋                                          黑盒子
#
#   d=1：2 個係數（β₀, β₁）→ 每個都能直接解讀
#   d=3：4 個係數 → 仍可理解但略複雜
#   d=15：16 個係數 → 個別係數失去意義，但曲線貼合更好
#
#   核心取捨：預測準確 vs 解釋能力，沒有免費午餐！
# ═══════════════════════════════════════════

| 方法 | 靈活度 | 可解釋性 | 說明 |
|------|--------|---------|------|
| 線性迴歸 | 低 | 高 | 每個係數直接解釋 |
| Lasso | 中低 | 高 | 自動選取重要變數 |
| GAM | 中 | 中 | 允許非線性但保持加法結構 |
| 決策樹 | 中高 | 中 | 可視化決策規則 |
| Bagging/Boosting | 高 | 低 | 集成方法，難以解釋 |
| SVM（非線性核） | 高 | 低 | 黑盒子 |
| 深度學習 | 極高 | 極低 | 數百萬參數 |

> **核心洞察**：即使目標是純預測，最靈活的模型也不一定最好——因為過度擬合！

---

## 六、監督學習 vs 非監督學習

### 6.1 監督學習（Supervised Learning）

對於每個觀測值 $i$，我們**同時擁有** $x_i$ 和 $y_i$。
目標：用 $x$ 預測 $y$，或理解 $x$ 與 $y$ 的關係。
→ 本書大部分內容屬於此類。

### 6.2 非監督學習（Unsupervised Learning）

對於每個觀測值 $i$，我們**只有** $x_i$，**沒有** $y_i$。
→ 沒有反應變數來「監督」分析！
目標：發現資料中的結構或模式（如分群）。

In [ ]:
# ============================================================
# Cell 9：非監督學習 — KMeans 分群
# 目的：展示「沒有 Y 標籤」時如何從 X 中發現隱藏結構
# 對應課本 Figure 2.8 — 分群是非監督學習的典型範例
# 核心差異：監督學習有 Y 來校正，非監督學習全憑 X 的內部結構
# ============================================================
from sklearn.cluster import KMeans  # K-means 分群演算法

np.random.seed(42)

# --- 模擬三群二維資料 ---
# 每群 50 個點，從不同的二維常態分布中抽樣
# 真實世界：我們只知道 X 座標，不知道哪個點屬於哪一群
g1 = np.random.multivariate_normal([2, 2], [[0.5, 0], [0, 0.5]], 50)   # 群 1：中心在 (2,2)
g2 = np.random.multivariate_normal([7, 8], [[0.5, 0], [0, 0.5]], 50)   # 群 2：中心在 (7,8)
g3 = np.random.multivariate_normal([5, 3], [[0.5, 0], [0, 0.5]], 50)   # 群 3：中心在 (5,3)
X_clus = np.vstack([g1, g2, g3])  # 垂直堆疊 → 150 個點 × 2 維

# --- K-means 分群 ---
# n_clusters=3：指定分成 3 群（假設我們知道有 3 群）
# n_init=10：從 10 個不同初始中心點開始，選 SSE 最小的結果
# 演算法自動迭代，直到群中心不再移動
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_clus)

# --- 繪圖 ---
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
# true_labels：我們知道真相（模擬時設定），但演算法不知道
true_labels = np.repeat([0, 1, 2], 50)
ax.scatter(X_clus[:, 0], X_clus[:, 1], c=true_labels, s=30, alpha=0.7,
           edgecolors='k', linewidth=0.5, cmap='Set1')
ax.set_xlabel('X₁')
ax.set_ylabel('X₂')
ax.set_title('Clustering: 無 y 值，僅根據 X 發現群體結構\n（Figure 2.8 概念）')
plt.show()

# --- 評估分群品質 ---
# Adjusted Rand Index (ARI)：比較分群結果與真實標籤
# ARI = 1 → 完美分群  |  ARI ≈ 0 → 約等於隨機分群
from sklearn.metrics import adjusted_rand_score
print(f"Adjusted Rand Index: {adjusted_rand_score(true_labels, km.labels_):.3f}")
print()
print("學習類型比較：")
print("  ├─ 監督學習：有 X 也有 Y → 學 X→Y 的映射關係")
print("  ├─ 非監督學習：只有 X → 從資料內部發現結構")
print("  └─ 半監督學習：部分有 Y、部分沒有 → 兩者混合利用")

還有**半監督學習**（semi-supervised learning）：部分資料有 y、部分沒有——本書不深入探討。

---

## 七、迴歸問題 vs 分類問題

| 類型 | 反應變數 Y | 例子 | 常用方法 |
|------|-----------|------|---------|
| **迴歸**（Regression） | 定量（數值連續） | 房價、銷售額、身高 | 線性迴歸 |
| **分類**（Classification） | 定性（類別） | 是否違約（是/否）、品牌選擇（A/B/C） | 邏輯斯迴歸 |

In [ ]:
# ============================================================
# Cell 10：迴歸 vs 分類 — 兩種監督學習的直觀對比
# 目的：一張圖同時展示迴歸（預測連續值）和分類（預測類別）
# 關鍵區別：Y 的資料型態決定分析方法
# ============================================================
from sklearn.linear_model import LogisticRegression     # 邏輯斯迴歸（雖然叫迴歸，但是分類方法！）
from sklearn.datasets import make_classification        # 方便生成分類資料

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ===== 左圖：迴歸（Regression）=====
# Y 是連續數值（ℝ），例如：房價 350.5 萬、銷售量 12.3 千單位
Xr = np.random.uniform(0, 10, 100).reshape(-1, 1)
yr = 3 + 2 * Xr.ravel() + np.random.normal(0, 3, 100)  # Y = 3 + 2X + ε，連續值
mr = LinearRegression().fit(Xr, yr)

# 畫出資料點和迴歸線
ax1.scatter(Xr, yr, c='red', alpha=0.5)
xp = np.linspace(0, 10, 100).reshape(-1, 1)
ax1.plot(xp, mr.predict(xp), 'b-', lw=2)
ax1.set_title('Regression: Y ∈ ℝ (連續數值)')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')

# ===== 右圖：分類（Classification）=====
# Y 是類別標籤 {0, 1}，例如：是否違約、是否點擊廣告
Xc, yc = make_classification(n_samples=200, n_features=2, n_redundant=0,
                              n_clusters_per_class=1, random_state=42)

# 邏輯斯迴歸：學習 P(Y=1 | X) 的機率，再用門檻值決定類別
# 注意：名字有「迴歸」，但輸出是類別 → 它是分類方法
mc = LogisticRegression().fit(Xc, yc)

# --- 畫出決策邊界 ---
# 建立網格點，對每個點做預測，再畫出決策區域（背景著色）
x_min, x_max = Xc[:, 0].min()-1, Xc[:, 0].max()+1
y_min, y_max = Xc[:, 1].min()-1, Xc[:, 1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), 
                      np.linspace(y_min, y_max, 200))
Z = mc.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# contourf：畫出填色區域（紅 = 類別 0 的區域，藍 = 類別 1 的區域）
ax2.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')

# scatter：畫出實際資料點，顏色代表真實類別
ax2.scatter(Xc[:, 0], Xc[:, 1], c=yc, edgecolor='k', s=30, cmap='RdYlBu')
ax2.set_title(f'Classification: Y ∈ {{0, 1}} (類別)\nAccuracy = {mc.score(Xc, yc):.2%}')
ax2.set_xlabel('X₁')
ax2.set_ylabel('X₂')

plt.suptitle('Regression vs Classification', fontsize=14)
plt.tight_layout()
plt.show()

# ════════════════ 選擇規則 ════════════════
# Y 是連續數值（ℝ）→ 迴歸方法
#   ├─ 線性迴歸、多項式迴歸、KNN 迴歸
#   └─ 輸出：一個實數值（例如：預測銷售量 = 15.3）
#
# Y 是類別標籤（{A, B, C} 或 {0, 1}）→ 分類方法
#   ├─ 邏輯斯迴歸、決策樹、SVM
#   └─ 輸出：一個類別標籤（例如：預測「會違約」）
#
# ⚠️ 注意：邏輯斯迴歸名字有「迴歸」但它是分類法！
#     → 它學的是 P(Y=1|X)，再轉成類別
# ═══════════════════════════════════════════

> 注意：邏輯斯迴歸雖然名字有「迴歸」，但它是**分類方法**（預測類別機率）。

---

## 八、實務應用要點

### 8.1 如何選擇統計學習方法？

```
我需要預測還是理解？
├── 預測為主 → 可用靈活模型（但要小心過度擬合！）
├── 推論為主 → 選可解釋模型（線性迴歸、Lasso）
└── 兩者兼顧 → 適中模型（GAM、決策樹）

Y 是數值還是類別？
├── 數值（連續） → 迴歸方法
└── 類別（離散） → 分類方法

我有 y 標籤嗎？
├── 有 → 監督學習（本書主軸）
└── 沒有 → 非監督學習（分群等）
```

### 8.2 常見陷阱

1. **過度擬合**：模型記住了訓練資料的雜訊，而非真實模式
2. **只用訓練誤差評估模型**：必須用獨立的測試資料！
3. **忽略不可約誤差**：即使完美估計 f，預測仍有誤差 Var(ε)

---

## 今日關鍵句

> **「統計學習的本質，就是從資料中估計 Y = f(X) + ε 中的未知函數 f——而我們的核心挑戰永遠是：在逼近真實 f 的同時，不被雜訊 ε 所誤導。」**

— ISLP §2.1, adapted

---

*下一節預告：2.2 評估模型準確度 — 偏差-變異數權衡、訓練 vs 測試誤差、貝氏分類器*